# Lesson 03 - Agentic Design Patterns

এই পাঠে, আমরা কার্যকর AI এজেন্ট তৈরি করার জন্য তিনটি মৌলিক ডিজাইন প্যাটার্ন অনুসন্ধান করব:

1. **সুস্পষ্ট এজেন্ট নির্দেশাবলী** — এমন নির্দিষ্ট, ভূমিকা-সংজ্ঞায়িত প্রম্পট তৈরি করা যা এজেন্টের আচরণ নির্দেশ করে
2. **Pydantic মডেল সহ কাঠামোবদ্ধ আউটপুট** — নিশ্চিত করা যে এজেন্টরা পূর্বানুমানযোগ্য, যাচাইবাছাই করা তথ্য ফেরত দেয়
3. **একক দায়িত্ব এজেন্ট** — ফোকাসড এজেন্ট ডিজাইন করা যারা প্রতিটি একটি কাজ ভালভাবে করে

আমরা প্রতিটি প্যাটার্ন একটি **ট্রাভেল ডেস্টিনেশন রেকমেন্ডার** পরিস্থিতিতে প্রয়োগ করব, পরপর একটি সিস্টেম তৈরি করব যা গন্তব্য প্রস্তাব করতে পারে, উপলব্ধতা পরীক্ষা করতে পারে, এবং লজিস্টিক্স পরিচালনা করতে পারে।


## সেটআপ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## প্যাটার্ন ১: স্পষ্ট এজেন্ট নির্দেশিকা

সবচেয়ে কার্যকর প্যাটার্নটি একই সাথে সবচেয়ে সাধারণ: আপনার এজেন্টের জন্য স্পষ্ট, বিস্তারিত নির্দেশিকা লেখা।

ভাল নির্দেশিকা নির্ধারণ করে:
- **কে** এজেন্ট (ব্যক্তিত্ব এবং ভাষা)
- **কি** এটি করা উচিত (ধাপে ধাপে দায়িত্ব)
- **কিভাবে** এটি আচরণ করা উচিত (সীমাবদ্ধতা এবং শৈলী)

নীচে, আমরা একটি ট্রাভেল কনসিয়ার্জ এজেন্ট তৈরি করি যার স্পষ্ট নির্দেশিকা প্রতিটি উত্তরের আকার নির্ধারণ করে।


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## প্যাটার্ন ২: পাইড্যান্টিক মডেল সহ কাঠামোবদ্ধ আউটপুট

ফ্রি-ফর্ম টেক্সট কথোপকথনের জন্য উপযোগী, কিন্তু পরবর্তী সিস্টেমগুলোকে কাঠামোবদ্ধ ডেটার প্রয়োজন হয়।  
**পাইড্যান্টিক মডেল** এবং একটি **টুল ফাংশন** জুড়েযে, আমরা করতে পারি:

- এজেন্টের আউটপুটের জন্য সঠিক একটি স্কিমা নির্ধারণ করা  
- স্বয়ংক্রিয়ভাবে প্রতিক্রিয়া যাচাই করা  
- এজেন্টের ফলাফলগুলো নির্ভরযোগ্যভাবে অ্যাপ্লিকেশন লজিকে অন্তর্ভুক্ত করা  

আমরা এমন একটি টুলও পরিচয় করিয়ে দিই যা গন্তব্যের বিস্তারিত তথ্য প্রদান করে, ফলে এজেন্ট তার পরামর্শ বাস্তব ডেটার উপর ভিত্তি করে দিতে পারে।


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## প্যাটার্ন ৩: একক দায়িত্ব এজেন্ট

জটিল কাজগুলি একাধিক মনোনিবেশিত এজেন্টের মধ্যে কাজ ভাগ করে নেওয়ার মাধ্যমে উপকার পায়, যাদের প্রত্যেকের একটি একক দায়িত্ব থাকে:

- একটি **গন্তব্য বিশেষজ্ঞ** যারা স্থান এবং প্রাপ্যতা সম্পর্কে জানেন
- একটি **লজিস্টিক্স পরিকল্পনাকারী** যারা ফ্লাইট, হোটেল এবং ভ্রমণ সূচি পরিচালনা করেন

এটি সফটওয়্যার ইঞ্জিনিয়ারিংয়ের *দায়িত্ব পৃথকীকরণের* নীতি প্রতিফলিত করে — প্রতিটি এজেন্ট স্বাধীনভাবে পরীক্ষা করা, রক্ষণাবেক্ষণ করা এবং উন্নত করা সহজ।


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## সারাংশ

এই পাঠে আমরা একটি ট্রাভেল রিকমেন্ডার পরিস্থিতিতে তিনটি এজেন্টিক ডিজাইন প্যাটার্ন প্রয়োগ করেছি:

| প্যাটার্ন | মূল ধারণা | সুবিধা |
|---|---|---|
| **স্পষ্ট নির্দেশাবলী** | পার্সোনা, দায়িত্ব এবং সীমাবদ্ধতা আগ্রিম নির্ধারণ করা | সঙ্গতিপূর্ণ, ব্র্যান্ডের সাথে মূলত এজেন্ট আচরণ |
| **গঠনমূলক আউটপুট** | উত্তর ফরম্যাট হিসেবে Pydantic মডেল ব্যবহার করা | বৈধকৃত, যন্ত্র-পাঠযোগ্য ফলাফল |
| **একক দায়িত্ব** | প্রতিটি এজেন্টকে একক ফোকাসড কাজ দেওয়া | পরীক্ষা করা, রক্ষণাবেক্ষণ এবং সংযোজন সহজ হয় |

এই প্যাটার্নগুলো প্রাকৃতিকভাবে সমন্বিত হয় — আপনি স্পষ্ট নির্দেশাবলী এবং গঠনমূলক আউটপুট একক দায়িত্বের এজেন্টের মধ্যে যোগ করে দৃঢ়, প্রোডাকশন-প্রস্তুত সিস্টেম তৈরি করতে পারেন।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**বিধিনিষেধ**:  
এই নথিটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা যথাসাধ্য সঠিকতার চেষ্টা করি, তবে স্বয়ংক্রিয় অনুবাদে ভুল বা অসম্পূর্ণতা থাকতে পারে। মূল নথিটি তার স্বত্বাধিকারী ভাষায়ই সর্বোত্তম এবং বিশ্বাসযোগ্য উৎস হিসেবে বিবেচনা করা উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানবরচিত অনুবাদের পরামর্শ দেওয়া হচ্ছে। এই অনুবাদের ব্যবহার থেকে উদ্ভূত কোনও ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়ী থাকব না।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
